# HadISD with PyEarthTools & XGBoost
HadISD is a global sub-daily dataset based on the ISD dataset from NOAA's NCEI. As well as station selection criteria, a suite of quality control tests has been run on the major climatological variables.

For this demonstration we are using version 3.4.0.2023f of the dataset, which can be downloaded here: https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/download.html

Before the dataset can be used, it must be converted to Zarr. Please refer to the HadISD_to_Zarr jupyter notebook for information on how to do this.

In [ ]:
import datetime
import numpy as np
import pandas as pd
from pathlib import Path

import pyearthtools.pipeline as petpipe
import pyearthtools.data as petdata
from pyearthtools.tutorial.HadisdDataClass import HadISDIndex

The following cell is used to run a configuration notebook that allows custom pipeline steps to be defined and included in the pipeline.

In [ ]:
%run ../HadISD_config.ipynb

## Creating an ordered list of all available HadISD station IDs
The  HadISDIndex class contains a method called "get_alL station_ids" that can be used to create a list of all available station IDS. This list can be used to select, for example, the first_ten stations from all stations. It can then be passedto select data from within the pipeline. It's also possible to pass a single station ID as a string to retrieve data for a single station.

In [ ]:
hadisd = HadISDIndex()
all_stations = hadisd.get_all_station_ids(Path("/Users/joelmiller/Projects/data/hadisd"))
all_stations_ordered = sorted(all_stations)
print(f"Total number of stations: {len(all_stations_ordered)}")

# Select first n stations
first_ten_stations = all_stations_ordered[:10]
print(first_ten_stations)

TODO:
- It would be good if a range of stations can be provided, or a boundary with stations inside
- Train test split should be done before normalising?
- How about before splitting into features and targets? 
- Train test split should be done after removing flagged data
- Make numpy conversion step work for single station. Add functionality for selecting range, selecting nearest neigbours, etc

## Pre-Training Data Pipeline
- Hadisd accepts "all" as an argument to get all stations.
- A single station can also be specified, e.g., "010010-99999".
- A list of station IDs can also be passed, e.g., ["010010-99999", "010014-99999"]

In [ ]:
data_prep_pipe = petpipe.Pipeline(
    petdata.archive.hadisd(station = first_ten_stations), #, variables = ["total_cloud_cover", "temperatures", "flagged_obs", "quality_control_flags"]), # Commented out variable selection whilst looking into bug in BaseTransforms
    SqueezeStationCoordinates(),
    petdata.transforms.values.AddFlaggedObs(flagged_labels),
    petdata.transforms.values.SetMissingToNaN(varname_val_map),
    petdata.transforms.variables.Drop("flagged_obs"),
    FeatureTargetSplit(target_vars="quality_control_flags"),
    (
        # branch 1: for X
        (
            # petdata.transforms.variables.Drop("high_cloud_cover"),
            petpipe.operations.xarray.conversion.ToNumpy(),
            MedianImputePerStation(),
            TransposeAndFlattenX(),

        ),  
        # branch 2: for y
        (
            petpipe.operations.xarray.conversion.ToNumpy(),
            SelectAndFlattenY(test_number=33)  # Select test from y and flatten it
            # Step 1 - Move features to the last axis, then flatten station and time
            # Step 2 - Reshape y to match X_flat

                 
        ),
        'map' # enables mapped branching
    ),
    # The above returns a tuple which can be used by the next step
    FilterNaNSamples(),
    #petdata.transofmrs.derive.WetBulbTemp(), # Just an idea for now. Should probably be in a separate pipeline where heavier processing is done
)
data_prep_pipe

In [ ]:
# Select a specific date to test the pipeline
# TODO: Figure out how to just give me all the data without specifying a date FIX THIS SOON
x, y = data_prep_pipe["1969-01-01T00"]
print (f"Shape of X: {x.shape}")
print (f"Shape of y: {y.shape}")

## Train Model using XGBoost with output from the pipeline

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Split data (stratify to preserve class imbalance in both sets)
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (len(y_train) - np.sum(y_train)) / np.sum(y_train)
#scale_pos_weight = num_zeros / num_ones 

# Create and fit model
classifier = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    objective='binary:logistic',
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss'
)
classifier.fit(X_train, y_train)

# Predict on test set
y_pred = classifier.predict(X_test)

## Confusion Matrix and Classification Report

In [ ]:
# compare predictions with true labels
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred)) 

# Plot confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns
def plot_confusion_matrix(cm, classes, title='Confusion Matrix', cmap=plt.cm.Blues):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

cm = confusion_matrix(y_test, y_pred)
plot_confusion_matrix(cm, classes=['0', '1'], title='Confusion Matrix for HadISD Quality Control Flags')